# Learning to rank (pointwise, pairwise, listwise) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Training on labels, preferences, or whole ordered lists
Learning to rank changes the loss to match the ranking decision. The five examples compute pointwise logistic loss, pairwise hinge/logistic preferences, a listwise softmax loss, and the gradient direction that separates relevant from irrelevant items.

In [ ]:
# 1) pointwise logistic loss treats each item independently
labels=np.array([1,0,1]); scores=np.array([2.0,0.5,1.0]); probs=sigmoid(scores)
loss=float(-np.mean(labels*np.log(probs)+(1-labels)*np.log(1-probs)))
plt.figure(figsize=(6,3)); plt.bar(['i0','i1','i2'],probs); plt.title(f'pointwise loss = {loss:.3f}')
assert abs(loss-0.47142222758043406)<1e-9

In [ ]:
# 2) pairwise hinge charges a relevant item not beating an irrelevant item by margin 1
s_pos=1.2; s_neg=0.7; loss=max(0,1-(s_pos-s_neg))
plt.figure(figsize=(6,3)); plt.bar(['positive','negative'],[s_pos,s_neg]); plt.axhline(s_neg+1,c='r',ls='--'); plt.title(f'hinge loss = {loss:.3f}')
assert abs(loss-0.5)<1e-12

In [ ]:
# 3) pairwise logistic is smooth: log(1+exp(-(s+ - s-)))
diffs=np.linspace(-2,4,100); losses=np.log1p(np.exp(-diffs)); val=float(np.log1p(np.exp(-0.5)))
plt.figure(figsize=(6,3)); plt.plot(diffs,losses); plt.scatter([0.5],[val],c='r'); plt.xlabel('score difference'); plt.ylabel('loss'); plt.title(f'loss at diff=.5 is {val:.3f}')
assert abs(val-0.4740769841801067)<1e-12

In [ ]:
# 4) listwise softmax loss normalizes over the whole ranked slate
scores=np.array([2.0,0.5,1.0]); target=0
p=np.exp(scores)/np.exp(scores).sum(); loss=float(-np.log(p[target]))
plt.figure(figsize=(6,3)); plt.bar(['i0','i1','i2'],p); plt.title(f'listwise prob(target)={p[target]:.3f}, loss={loss:.3f}')
assert abs(loss-0.46436878410794485)<1e-9

In [ ]:
# 5) gradient pushes target up and the rest down under listwise loss
scores=np.array([2.0,0.5,1.0]); target=0; p=np.exp(scores)/np.exp(scores).sum(); grad=p.copy(); grad[target]-=1
plt.figure(figsize=(6,3)); plt.bar(['i0','i1','i2'],grad); plt.axhline(0,c='k'); plt.title('softmax gradient')
assert grad[0]<0 and grad[1]>0 and abs(grad.sum())<1e-12